# Train a model on Colab

1. **Runtime → Change runtime type → A100 GPU** (or L4 to use fewer compute units). Don't use CPU or TPU.
2. Run the cells in order.
3. If Colab disconnects, reconnect and run the **Setup** cell and your training cell again. Training **resumes** from the last checkpoint, which is saved in Google Drive.

Edit code on your laptop and push to GitHub; don't edit code here (changes made in Colab are lost).

## Setup

In [ ]:
BRANCH = "model-harness"   # change to "main" once the PR is merged
REPO = "https://github.com/alliemi/cis5300-finalproject.git"

import os
os.chdir("/content")
if not os.path.exists("cis5300-finalproject"):
    !git clone -b {BRANCH} {REPO}
%cd /content/cis5300-finalproject
!git fetch -q && git checkout -q {BRANCH} && git pull -q
!test -f data/train.jsonl || tar -xzf data.tar.gz

# Keep runs/ (checkpoints, logs, predictions) in Google Drive so they survive disconnects.
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/cis5300-runs && ln -sfn /content/drive/MyDrive/cis5300-runs runs
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Timing run (about a minute)

200 steps, to measure tokens/second on this GPU. The log prints the estimated minutes per epoch.

In [ ]:
!python train.py --config configs/linear.json --run runs/timing-linear-$(nvidia-smi --query-gpu=name --format=csv,noheader | tr ' ' '-') --max-steps 200 --no-final-eval

## Full training run

Uses the shared recipe in `configs/base.json`. When it finishes, it predicts the full dev set with the best checkpoint and writes `dev_scores.json` (via `score.py`) into the run folder. Re-running the same command resumes.

In [ ]:
!python train.py --config configs/linear.json --run runs/linear-s0 --seed 0

## Compare finished runs

In [ ]:
!python summarize_runs.py